# Stock Price Prediction

## Problem Statement
Stock price prediction is a complex task due to the high volatility and randomness of financial markets. Traditional statistical models often fail to capture long-term dependencies in stock price movements. Deep learning, particularly Long Short-Term Memory (LSTM) networks, has shown great potential in handling time-series forecasting by learning from historical patterns.

## Step-by-Step

## Precheck

In [37]:
!python3 -m pip config set global.break-system-packages true

In [38]:
!python3 -m pip install ipykernel -U --force-reinstall --break-system-packages --no-warn-script-location

In [39]:
!python3 -V && pip3 -V

### Install dependencies

In [49]:
!pip3 install -U yfinance torch numpy pandas scikit-learn matplotlib ipython-autotime pip-system-certs certifi

%load_ext autotime

### Import Libraries 

In [50]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf   # ref: https://yfinance-python.org/
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

### Load & Preprocess Stock Data

In [51]:
# BEGIN: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED
import os, ssl
# BEGIN: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED
if (not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None)):
    ssl._create_default_https_context = ssl._create_unverified_context
# END: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED

#### Define variables

In [52]:
# Download stock data for Apple (AAPL)
stock='AAPL'
todaysDate = pd.Timestamp.today().date().strftime('%Y-%m-%d')

print(f"Downloading stock data for '{stock}' from '1990-01-01' to '{todaysDate}' ")

# yf.enable_debug_mode()
# data = yf.download(stock) # start='1990-01-01', end=todaysDate)
data = yf.Ticker(stock).history(period='max')
data

In [44]:

# Extract 'Close' prices and scale them
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[['Close']])

# Create time-series sequences
def create_sequences(data, seq_length=50):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 50  # Lookback period
X, y = create_sequences(data_scaled, seq_length)

# Train-test split (80-20)
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert to PyTorch tensors
X_train_tensor, X_test_tensor = torch.Tensor(X_train), torch.Tensor(X_test)
y_train_tensor, y_test_tensor = torch.Tensor(y_train), torch.Tensor(y_test)
